In [ ]:
import pandas as pd
import numpy as np
import pickle
import torch

from sbi.utils import BoxUniform
from scipy.integrate import odeint
from sbi.utils import MultipleIndependent

import matplotlib.pyplot as plt

_ = np.random.seed(0)      

In [ ]:
with open('../../Data/Model2/M2_dataset.pkl', 'rb') as handle:
    sim_traj = pickle.load(handle)

with open('../../Data/Model2/M2_params.pkl', 'rb') as handle:
    sim_params = pickle.load(handle)
    
with open('../../Data/Model2/sim_dataset.pkl', 'rb') as f:
    simulation_dataset = pickle.load(f)

In [ ]:
param_names = ["beta_I", "beta_H", "kappa", "alpha", "gamma_I", "delta_I", "gamma_H", "delta_H"]

In [ ]:
def poisson_noise(simulation):
    
    with_noise = np.random.poisson(np.maximum(simulation, 1e-6))

    return with_noise

In [ ]:
traj   = torch.tensor(poisson_noise(sim_traj.numpy()), dtype=torch.float32)
params = sim_params

In [ ]:
from sbi.utils import BoxUniform
from scipy.integrate import odeint
from sbi.utils import MultipleIndependent

import torch
from torch.distributions import Uniform, Exponential, Cauchy

_ = torch.manual_seed(0)

device = "cuda:0" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

In [ ]:
import torch.nn as nn
from sbi.inference import NPE
from sbi.neural_nets import posterior_nn

class LSTMembedding(nn.Module):
    def __init__(self, input_dim=1, hidden_dim=128, output_dim=30, num_layers=1,bidirectional=True):
        super().__init__()

        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, bidirectional=bidirectional, batch_first = True)
        self.fc = nn.Linear(hidden_dim * (2 if bidirectional else 1), output_dim)
        self.tanh = nn.Tanh()
        self.dropout = nn.Dropout(0.1)

    def forward(self, x):
        lstm_out, _ = self.lstm(x)  
        last_hidden = lstm_out[:, -1, :]
        last_hidden = self.dropout(last_hidden)
        out = self.fc(last_hidden)   
        return out

In [ ]:
embedding_net =LSTMembedding(input_dim=3, hidden_dim=128, output_dim=30).to(device)
embedding_net

In [ ]:
import torch.nn as nn
from sbi.inference import NPE
from sbi.neural_nets import posterior_nn

In [ ]:
low  = torch.tensor([0.6, 0.6, 0.1, 0.1, 0.05, 0.001, 0.05, 0.01])
high = torch.tensor([1.5, 1.5, 0.5, 0.3, 0.4, 0.02, 0.2, 0.3])

prior = BoxUniform(low=low, high=high)

In [ ]:
xs     = traj
thetas = params

mean = xs.mean(axis=(0,1))
std = xs.std(axis=(0,1))+1e-6

xs_norm = (xs-mean)/std
xs_norm = torch.tensor(xs_norm, dtype=torch.float32)

In [ ]:
neural_posterior = posterior_nn(model='maf', embedding_net=embedding_net)

inference = NPE(density_estimator=neural_posterior,device=device)

# training
density_estimator=inference.append_simulations(thetas, xs_norm).train(training_batch_size=256)

posterior = inference.build_posterior(density_estimator)


In [ ]:
posterior_samples=[]

for i, sim in enumerate(simulation_dataset):  
    x_obs = torch.as_tensor(sim['poisson'], dtype=torch.float32).to(device)
    x_obs_norm = (x_obs-mean)/std
    
    samples = posterior.sample((10000,), x=x_obs_norm)
    samples_df=pd.DataFrame(samples.cpu().numpy(),columns=param_names)
    posterior_samples.append(samples_df)